In [61]:
import sys
sys.path.append('..')
import pandas as pd
from src.stock_data_loader import load_stock, fix_dtypes, handle_missing
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import matplotlib.pyplot as plt
import numpy as np

analyzer = SentimentIntensityAnalyzer()

Load and clean news

In [42]:
df_news = pd.read_csv('../data/raw/raw_analyst_ratings.csv', index_col=0)

df_news['date'] = pd.to_datetime(df_news['date'].str.replace(r'-\d{2}:\d{2}$', '', regex=True))

# Extract just the date (remove time)
df_news['news_date'] = df_news['date'].dt.date

print(f"Loaded {len(df_news):,} news articles")
print(f"Date range: {df_news['news_date'].min()} to {df_news['news_date'].max()}")

Loaded 1,407,328 news articles
Date range: 2009-02-14 to 2020-06-11


Filter to ONLY for 5 Stocks and Rename FB to META

In [43]:
# Rename FB to META (because stock data uses META.csv)
df_news.loc[df_news['stock'] == 'FB', 'stock'] = 'META'

# Define your 5 target stocks
target_stocks = ['AAPL', 'AMZN', 'GOOG', 'META', 'NVDA']

# Filter news to only these 5 stocks
df_news = df_news[df_news['stock'].isin(target_stocks)]

print(f"\nNews after filtering to 5 stocks: {len(df_news):,} articles")
print("\nCount per stock:")
print(df_news['stock'].value_counts())


News after filtering to 5 stocks: 5,444 articles

Count per stock:
stock
NVDA    3146
GOOG    1199
AAPL     441
META     380
AMZN     278
Name: count, dtype: int64


Load stocks and collect trading days

In [52]:
tickers = ['AAPL', 'AMZN', 'GOOG', 'META', 'NVDA']
all_trading_days = set()
all_stock_data = {}

for ticker in tickers:
    df = load_stock(ticker)
    df = fix_dtypes(df)
    df = handle_missing(df)

    all_stock_data[ticker] = df
    
    # Add all trading days from this stock
    all_trading_days.update(df['Date'].dt.date)
    
    print(f"{ticker}: {len(df)} rows, {df['Date'].min().date()} to {df['Date'].max().date()}")

print(f"\nTotal unique trading days from all 5 stocks: {len(all_trading_days):,}")

AAPL: 3774 rows, 2009-01-02 to 2023-12-29
AMZN: 3774 rows, 2009-01-02 to 2023-12-29
GOOG: 3774 rows, 2009-01-02 to 2023-12-29
META: 2923 rows, 2012-05-18 to 2023-12-29
NVDA: 3774 rows, 2009-01-02 to 2023-12-29

Total unique trading days from all 5 stocks: 3,774


Get trading days

In [45]:
def get_next_trading_day(date_obj, trading_days):
    """
    If date is a weekend or holiday, find the next trading day.
    Example: Saturday → Monday, Holiday → next open day
    """
    date_obj = pd.to_datetime(date_obj).date()
    while date_obj not in trading_days:
        date_obj = date_obj + pd.Timedelta(days=1)
    return date_obj

# Test the function
test_date = pd.to_datetime('2020-03-07').date() 
print(f"Test: {test_date} → {get_next_trading_day(test_date, all_trading_days)}")

Test: 2020-03-07 → 2020-03-09


Apply Alignment to News

In [49]:
# Align: keep date if trading day, otherwise shift to next trading day
df_news['trading_day'] = df_news['news_date'].apply(
    lambda x: x if x in all_trading_days else get_next_trading_day(x, all_trading_days)
)
print(f"Original unique news dates: {df_news['news_date'].nunique():,}")
print(f"Unique trading days after alignment: {df_news['trading_day'].nunique():,}")

Original unique news dates: 1,318
Unique trading days after alignment: 1,222


### Sentiment Analysis

In [50]:
financial_terms = {
    # Strong moves
    "crashes": -1.5,
    "plunges": -1.5,
    "tumbles": -1.2,
    "slumps": -1.0,
    "surges": 1.5,
    "soars": 1.5,
    "jumps": 1.2,
    "climbs": 1.0,
    
    # Analyst actions
    "upgrade": 1.0,
    "downgrade": -1.0,
    "buy": 0.6,
    "sell": -0.6,
    "raises": 0.8,
    "lowers": -0.8,
    "maintains": 0.0,
    
    # Earnings
    "beat": 1.0,
    "miss": -1.0,
    "profit": 0.6,
    "loss": -0.6,
    
    # Price targets
    "price target": 0.0,
}

analyzer.lexicon.update(financial_terms)
# Now test 
test = "Apple stock CRASHES!!! Down 10%"
print(analyzer.polarity_scores(test)['compound'])

-0.626


In [51]:
# Apply sentiment to each headline
df_news['sentiment'] = df_news['headline'].fillna('').apply(
    lambda x: analyzer.polarity_scores(str(x))['compound']
)

print("Sentiment scores added")
print(f"Total headlines: {len(df_news):,}")
print(f"Mean sentiment: {df_news['sentiment'].mean():.4f}")
print(f"Std sentiment: {df_news['sentiment'].std():.4f}")

Sentiment scores added
Total headlines: 5,444
Mean sentiment: 0.0941
Std sentiment: 0.3031


### Sentiment Analysis: VADER

**Tool Selection:** VADER (Valence Aware Dictionary for sEntiment Reasoning) was chosen because it handles:
- Intensity (ALL CAPS, "!!!")
- Negation ("not good")
- Financial terms via custom lexicon extension

**Method:** Extended VADER with financial terms (surges, crashes, upgrade, beat, miss, etc.) using conservative intensity values (±1.8 max). Each headline receives a compound score from -1 (negative) to +1 (positive).

**Results:** 5,444 headlines analyzed | Mean: 0.094 | Std: 0.303

### Calculate Daily Stock Returns

In [53]:
# Calculate returns for all stocks
for ticker in all_stock_data:
    all_stock_data[ticker]['return_pct'] = all_stock_data[ticker]['Close'].pct_change() * 100

# Quick verification
print("Mean daily returns:")
for ticker, df in all_stock_data.items():
    print(f"  {ticker}: {df['return_pct'].mean():.3f}%")

Mean daily returns:
  AAPL: 0.129%
  AMZN: 0.130%
  GOOG: 0.091%
  META: 0.108%
  NVDA: 0.188%


### Aggregate and Correlate

In [57]:
# Aggregate: average sentiment per stock per day
daily_sentiment = df_news.groupby(['stock', 'trading_day'])['sentiment'].mean().reset_index()
daily_sentiment.columns = ['stock', 'date', 'avg_sentiment']

print(f"Original: {len(df_news)} rows")
print(f"Aggregated: {len(daily_sentiment)} rows")
print(daily_sentiment.head())

Original: 5444 rows
Aggregated: 1659 rows
  stock        date  avg_sentiment
0  AAPL  2020-03-09      -0.302067
1  AAPL  2020-03-10      -0.081238
2  AAPL  2020-03-11      -0.011086
3  AAPL  2020-03-12      -0.230340
4  AAPL  2020-03-13      -0.000273


In [58]:
# Extract returns from your existing data
stock_returns = []

for ticker, df in all_stock_data.items():
    temp = df[['Date']].copy()
    temp['stock'] = ticker
    temp['return_pct'] = df['return_pct']  # Already calculated
    temp['date'] = pd.to_datetime(temp['Date']).dt.date
    stock_returns.append(temp)

stock_returns = pd.concat(stock_returns, ignore_index=True)

print(stock_returns.head())

        Date stock  return_pct        date
0 2009-01-02  AAPL         NaN  2009-01-02
1 2009-01-05  AAPL    4.220416  2009-01-05
2 2009-01-06  AAPL   -1.649399  2009-01-06
3 2009-01-07  AAPL   -2.160860  2009-01-07
4 2009-01-08  AAPL    1.856959  2009-01-08


In [ ]:
# Merge sentiment and returns
sentiment_returns = pd.merge(
    daily_sentiment,
    stock_returns[['stock', 'date', 'return_pct']],
    on=['stock', 'date'],
    how='inner'
)

# Calculate Pearson correlation
correlation = sentiment_returns['avg_sentiment'].corr(sentiment_returns['return_pct'])

print(f"Pearson correlation: {correlation:.4f}")

Pearson correlation: 0.2246


In [60]:
# Per-stock correlation
print("\n" + "=" * 50)
print("PER-STOCK CORRELATION")
print("=" * 50)

stock_correlations = []

for stock in sentiment_returns['stock'].unique():
    stock_data = sentiment_returns[sentiment_returns['stock'] == stock]
    corr = stock_data['avg_sentiment'].corr(stock_data['return_pct'])
    stock_correlations.append({'stock': stock, 'correlation': corr})
    print(f"{stock}: {corr:.4f}")

# Summary
print("\n" + "=" * 50)
print("SUMMARY")
print("=" * 50)
corr_df = pd.DataFrame(stock_correlations)
print(f"Overall (all stocks combined): {correlation:.4f}")
print(f"Range: {corr_df['correlation'].min():.4f} to {corr_df['correlation'].max():.4f}")


PER-STOCK CORRELATION
AAPL: 0.1697
AMZN: 0.1030
GOOG: 0.1884
META: 0.4921
NVDA: 0.2240

SUMMARY
Overall (all stocks combined): 0.2246
Range: 0.1030 to 0.4921
